# Versorgungsgrad

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
BASE_PATH = '../data'
SHAPEFILE_PATH = f'{BASE_PATH}/vg250_01-01.utm32s.shape.ebenen/vg250_ebenen_0101/VG250_KRS.shp'
KBV_PATH = f'{BASE_PATH}/KBV_Daten.csv'

# --- COLUMN MAPPING ---
KBV_ID_COL = 'AGS' 
KBV_VALUE_COL = 'Versorgungsgrad' 
KBV_DISTRICT_COL = 'REG_BEZIRK'

def clean_german_number(value):
    if isinstance(value, str):
        clean_val = value.replace('.', '').replace(',', '.').replace('%', '').replace('"', '').strip()
        try:
            return float(clean_val)
        except ValueError:
            return None
    return value

def main():
    print("--- START: DATA PROCESSING (CONTINUOUS SCALE) ---")

    try:
        # --- 1. LOAD DATA ---
        if not os.path.exists(SHAPEFILE_PATH):
            raise FileNotFoundError(f"Shapefile not found: {SHAPEFILE_PATH}")

        geo_data = gpd.read_file(SHAPEFILE_PATH)
        kbv_data = pd.read_csv(KBV_PATH, sep=',', encoding='utf-8', na_values=['#NV'])

        # --- 2. CLEAN DATA ---
        kbv_clean = kbv_data.copy()
        kbv_clean = kbv_clean.dropna(subset=[KBV_ID_COL]).copy()
        
        # Format AGS (9771.0 -> "09771")
        kbv_clean[KBV_ID_COL] = kbv_clean[KBV_ID_COL].astype(str).str.split('.').str[0].str.zfill(5)
        
        # Clean numeric values
        kbv_clean[KBV_VALUE_COL] = kbv_clean[KBV_VALUE_COL].apply(clean_german_number)

        # --- 3. MERGE ---
        merged_data = geo_data.merge(
            kbv_clean, 
            left_on='AGS',       
            right_on=KBV_ID_COL, 
            how='inner'
        ).drop_duplicates(subset='AGS')

        # --- 4. FILTER (NORTHERN BAVARIA) ---
        north_bavaria_districts = ['Oberfranken', 'Mittelfranken', 'Unterfranken', 'Oberpfalz']
        merged_data[KBV_DISTRICT_COL] = merged_data[KBV_DISTRICT_COL].astype(str).str.strip()
        plot_data = merged_data[merged_data[KBV_DISTRICT_COL].isin(north_bavaria_districts)].copy()
        
        if len(plot_data) == 0:
            print("Warning: Filter returned empty. Showing all data.")
            plot_data = merged_data

        # --- 5. VISUALIZATION (CONTINUOUS) ---
        print("\nCreating map with continuous scale for correlation check...")
        
        # Set figure size
        fig, ax = plt.subplots(figsize=(12, 12))

        # Create axis for legend at the top
        cax = fig.add_axes([0.3, 0.92, 0.4, 0.02])

        # Plotting continuous scale
        # We use 'YlGnBu' (Yellow-Green-Blue) to contrast with Population Density (usually Yellow-Orange-Red)
        # This makes visual comparison easier: Dark Blue here vs Dark Red there.
        plot_data.plot(
            column=KBV_VALUE_COL,
            cmap='YlGnBu',              # Continuous gradient: Light Yellow (Low) -> Dark Blue (High)
            linewidth=0.8,
            edgecolor='0.8',            # Light grey borders
            legend=True,
            cax=cax,                    # Use custom axis
            legend_kwds={
                'label': "Psychotherapist Provision (%)",
                'orientation': 'horizontal'
            },
            missing_kwds={'color': 'lightgrey'},
            ax=ax
        )

        ax.set_axis_off()
        fig.suptitle('Psychotherapist Provision: Northern Bavaria (%)', fontsize=16, y=0.97)
        
        plt.tight_layout(rect=[0, 0, 1, 0.9])
        plt.show()

    except Exception as e:
        print(f"\nERROR: {e}")

if __name__ == "__main__":
    main()

# Wartezeiten

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- CONFIGURATION ---
BASE_PATH = '../data'
SHAPEFILE_PATH = f'{BASE_PATH}/vg250_01-01.utm32s.shape.ebenen/vg250_ebenen_0101/VG250_KRS.shp'
# Use KBV file as it contains the 'Wartezeit genau' column
KBV_PATH = f'{BASE_PATH}/KBV_Daten.csv' 

# IMPORTANT: Exact column name from KBV_Daten.csv
# "Wartezeit genau" contains data in days (e.g., 113, 100)
TARGET_COLUMN = 'Wartezeit genau' 

def clean_german_number(value):
    """Converts a string with a German number (or #NV) to float."""
    if pd.isna(value) or value == '#NV' or value == '':
        return np.nan
    if isinstance(value, str):
        # Remove dots (thousands separator) and replace comma with dot
        clean_val = value.replace('.', '').replace(',', '.')
        try:
            return float(clean_val)
        except ValueError:
            return np.nan
    return float(value)

def main():
    print("--- START: STATIC MAP GENERATION (Northern Bavaria + Oberpfalz - KBV Data) ---")

    try:
        # --- 1. LOAD DATA ---
        if not os.path.exists(SHAPEFILE_PATH):
            raise FileNotFoundError(f"Shapefile not found: {SHAPEFILE_PATH}")
        
        if not os.path.exists(KBV_PATH):
             raise FileNotFoundError(f"KBV Data file not found: {KBV_PATH}")

        print("Loading Shapefile...")
        geo_data = gpd.read_file(SHAPEFILE_PATH)

        print(f"Loading KBV Data from {KBV_PATH}...")
        # KBV file is usually comma-separated, but could be semicolon.
        # Based on snippet, it uses commas. If error -> try sep=';'
        kbv_data = pd.read_csv(KBV_PATH, sep=',', encoding='utf-8', dtype={'AGS': str})

        # --- 2. FILTER GEOMETRY (Northern Bavaria + Oberpfalz) ---
        print("Filtering Geometry for Northern Bavaria + Oberpfalz (093, 094, 095, 096)...")
        # Added code 093 (Oberpfalz) to Franconia (094, 095, 096)
        north_bavaria_geo = geo_data[geo_data['AGS'].str.startswith(('093', '094', '095', '096'))].copy()
        
        if len(north_bavaria_geo) == 0:
             print("⚠️ Warning: No regions found for Northern Bavaria/Oberpfalz. Using all regions.")
             north_bavaria_geo = geo_data.copy()

        # --- 3. CLEAN CSV DATA ---
        print(f"Cleaning data for column: '{TARGET_COLUMN}'...")
        
        # Work with a copy of the data
        data_clean = kbv_data.copy()
        
        # Ensure AGS is in the correct format (5 chars)
        # Snippet shows AGS exists, e.g., "09771"
        data_clean['AGS'] = data_clean['AGS'].astype(str).str.zfill(5)

        # Clean target column
        if TARGET_COLUMN in data_clean.columns:
            data_clean[TARGET_COLUMN] = data_clean[TARGET_COLUMN].apply(clean_german_number)
        else:
            print(f"⚠️ Column '{TARGET_COLUMN}' not found! Found columns: {data_clean.columns.tolist()}")
            print("Generating dummy data for visualization test...")
            np.random.seed(42)
            data_clean[TARGET_COLUMN] = np.random.randint(60, 180, size=len(data_clean)) # Days (2-6 months)

        # Remove rows without data (optional, merge left preserves geometry)
        # data_clean = data_clean.dropna(subset=[TARGET_COLUMN])

        # --- 4. MERGE ---
        print("Merging Geodata and KBV Data...")
        final_data = north_bavaria_geo.merge(
            data_clean, 
            on='AGS', 
            how='left'
        )

        # --- 5. VISUALIZATION ---
        print("Creating map...")
        
        # Figure size
        fig, ax = plt.subplots(figsize=(10, 12))

        # --- SCALE ON TOP (Horizontal) ---
        cax = fig.add_axes([0.25, 0.86, 0.5, 0.02]) 

        # Calculate min/max for the scale to go from smallest to largest value
        vmin = final_data[TARGET_COLUMN].min()
        vmax = final_data[TARGET_COLUMN].max()
        print(f"Scale range (Days): {vmin} to {vmax}")

        # Plot map
        final_data.plot(
            column=TARGET_COLUMN,
            cmap='YlOrRd',      # Yellow -> Orange -> Red
            linewidth=0.4,
            edgecolor='0.5',
            legend=True,
            cax=cax,
            vmin=vmin,          # Explicit minimum
            vmax=vmax,          # Explicit maximum
            legend_kwds={
                'label': f"Wartezeit in Tagen ({TARGET_COLUMN})", # IN DAYS
                'orientation': 'horizontal'
            },
            missing_kwds={'color': '#f0f0f0', 'label': 'Keine Daten'},
            ax=ax
        )

        ax.set_axis_off()

        # Titles
        plt.figtext(0.5, 0.92, 'Wartezeit auf einen Psychotherapieplatz', 
                   fontsize=18, fontweight='bold', ha='center', color='#333333')
        
        plt.figtext(0.5, 0.90, 'Region: Nordbayern und Oberpfalz', 
                   fontsize=12, ha='center', color='#555555')

        # Compact layout
        # Leave space at the top for titles and scale (top=0.85)
        plt.tight_layout(rect=[0, 0, 1, 0.85])
        
        output_file = 'northern_bavaria_oberpfalz_map.png'
        # plt.savefig(output_file, dpi=300, bbox_inches='tight') # Uncomment to save
        plt.show()
        print("Done.")

    except Exception as e:
        print(f"\n❌ ERROR: {e}")

if __name__ == "__main__":
    main()